In [1]:
import os 
from PIL import Image 
import numpy as np 
import tensorflow as tf 

In [2]:
lr_dir = r"C:/Users/91995/Downloads/div2k/DIV2K_valid_HR/DIV2K_valid_HR"
hr_dir = r"C:/Users/91995/Downloads/div2k/DIV2K_valid_HR/DIV2K_valid_HR"

In [3]:
scale_size = 2
HR_SIZE = (400, 400)
LR_SIZE = (200, 200)
patch_size = 10  
stride = 5

In [4]:
import os
import numpy as np
import tensorflow as tf
from PIL import Image
import cv2
def load_images_from_dir(directory, target_size=None):
    images = []
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    
    for filename in os.listdir(directory):
        # Check if file has valid image extension
        if os.path.splitext(filename.lower())[1] in valid_extensions:
            img_path = os.path.join(directory, filename)
            try:
                img = Image.open(img_path).convert('RGB')
                img_array = np.array(img, dtype=np.uint8)
                
                # Ensure image is 3D (height, width, channels)
                if len(img_array.shape) != 3:
                    continue
                    
                img_tensor = tf.convert_to_tensor(img_array, dtype=tf.uint8)
                img_tensor = tf.cast(img_tensor, tf.float32) / 255.0
                
                if target_size:
                    img_tensor = tf.image.resize(img_tensor, target_size, method='area')
                
                img_array = tf.cast(img_tensor * 255.0, tf.uint8).numpy()
                img_yuv = cv2.cvtColor(img_array, cv2.COLOR_RGB2YUV)
                images.append(img_yuv)
            except Exception as e:
                print(f"Error loading {filename}: {str(e)}")
                continue
    
    if not images:
        raise ValueError(f"No valid images found in directory: {directory}")
        
    return np.array(images, dtype=np.uint8)

In [5]:
lr_images = load_images_from_dir(lr_dir, target_size=(200, 200))  
hr_images = load_images_from_dir(hr_dir, target_size=(400, 400)) 

In [6]:
def process_input(input, input_size, upscale_factor):
    input = tf.cast(input, tf.float32) / 255.0
    lr_image = tf.image.resize(input, [input_size, input_size], method="area")
    return lr_image

def process_target(input):
    input = tf.cast(input, tf.float32) / 255.0
    hr_image = tf.image.resize(input, HR_SIZE, method='bilinear')
    return hr_image

def preprocess_image(data):
    hr_image = process_target(data)
    lr_image = process_input(data, LR_SIZE[0], scale_size)
    return lr_image, hr_image

def create_patches(image, patch_size, stride):
    # Ensure input is 4D (batch, height, width, channels)
    if len(image.shape) == 3:
        image = tf.expand_dims(image, 0)
    
    patches = tf.image.extract_patches(
        images=image,
        sizes=[1, patch_size, patch_size, 1],
        strides=[1, stride, stride, 1],
        rates=[1, 1, 1, 1],
        padding='VALID'
    )
    
    # Reshape patches to (num_patches, patch_size, patch_size, channels)
    patches = tf.reshape(patches, [-1, patch_size, patch_size, image.shape[-1]])
    return patches

def apply_preprocessing_on_local_data(lr_images, hr_images):
    if len(lr_images) != len(hr_images):
        raise ValueError("Number of LR and HR images must match")
    
    lr_patches_list = []
    hr_patches_list = []
    
    for i, (lr_image, hr_image) in enumerate(zip(lr_images, hr_images)):
        try:
            # Preprocess images
            lr_processed, hr_processed = preprocess_image(hr_image)
            
            # Create patches
            lr_patches = create_patches(lr_processed, patch_size, stride)
            hr_patches = create_patches(hr_processed, patch_size * scale_size, stride * scale_size)
            
            # Verify patches were created successfully
            if tf.shape(lr_patches)[0] > 0 and tf.shape(hr_patches)[0] > 0:
                lr_patches_list.append(lr_patches)
                hr_patches_list.append(hr_patches)
            else:
                print(f"Warning: No patches created for image {i}")
                
        except Exception as e:
            print(f"Error processing image {i}: {str(e)}")
            continue
    
    if not lr_patches_list or not hr_patches_list:
        raise ValueError("No valid patches were created from any images")
    
    # Concatenate all patches
    lr_patches = tf.concat(lr_patches_list, axis=0)
    hr_patches = tf.concat(hr_patches_list, axis=0)
    
    return lr_patches, hr_patches


In [7]:
try:
    # Load images
    lr_images = load_images_from_dir(lr_dir, target_size=LR_SIZE)
    hr_images = load_images_from_dir(hr_dir, target_size=HR_SIZE)
    
    print(f"Loaded {len(lr_images)} LR images and {len(hr_images)} HR images")
    
    # Create patches
    lr_patches, hr_patches = apply_preprocessing_on_local_data(lr_images, hr_images)
    
    print(f"Created {tf.shape(lr_patches)[0]} LR patches and {tf.shape(hr_patches)[0]} HR patches")
    
except Exception as e:
    print(f"Error in main processing: {str(e)}")

Loaded 100 LR images and 100 HR images
Created 152100 LR patches and 152100 HR patches


In [8]:
lr_patches, hr_patches = apply_preprocessing_on_local_data(lr_images, hr_images)
#print("Low-res patches shape:", lr_patches.shape)
#print("High-res patches shape:", hr_patches.shape)

In [9]:
def psnr(y_true, y_pred):
    max_pixel = 1.0
    return tf.image.psnr(y_true, y_pred, max_val=max_pixel)

class DepthToSpace(tf.keras.layers.Layer):
    def __init__(self, scale, **kwargs):
        super(DepthToSpace, self).__init__(**kwargs)
        self.scale = scale

    def call(self, inputs):
        return tf.nn.depth_to_space(inputs, self.scale)

def leaky_relu_activation(x):
    leaky_relu_out = tf.keras.layers.LeakyReLU(alpha=0.2)(x)
    linear_out = x
    return leaky_relu_out + linear_out

class SinglePixelAttention(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(SinglePixelAttention, self).__init__(**kwargs)
        
    def call(self, inputs):
        mean_values = tf.reduce_mean(inputs, axis=-1)
        max_indices = tf.argmax(mean_values, axis=-1)
        attention_mask = tf.one_hot(max_indices, depth=tf.shape(inputs)[1])
        attention_mask = tf.expand_dims(attention_mask, axis=-1)
        attention_output = inputs * attention_mask
        return attention_output

In [10]:
def build_espcn_model_with_skip_connections(input_shape, scale_size):
    inputs = tf.keras.layers.Input(shape=input_shape)
    conv1 = tf.keras.layers.Conv2D(16, 5, padding='same', kernel_initializer=tf.keras.initializers.HeNormal())(inputs)
    conv1 = tf.keras.layers.LeakyReLU(alpha=0.2)(conv1)
    conv2 = tf.keras.layers.Conv2D(16, 3, padding='same', kernel_initializer=tf.keras.initializers.HeNormal())(conv1)
    conv2 = tf.keras.layers.LeakyReLU(alpha=0.2)(conv2)
    skip1 = tf.keras.layers.Add()([conv1, conv2])
    conv3 = tf.keras.layers.Conv2D(16, 2, padding='same', kernel_initializer='orthogonal')(skip1)
    conv3 = tf.keras.layers.Lambda(leaky_relu_activation)(conv3)
    conv3 = SinglePixelAttention()(conv3)
    skip2 = tf.keras.layers.Add()([skip1, conv3])
    conv4 = tf.keras.layers.Conv2D(3 * (scale_size ** 2), 3, padding='same', kernel_initializer='orthogonal')(skip2)
    conv4 = tf.keras.layers.LeakyReLU(alpha=0.2)(conv4)
    outputs = DepthToSpace(scale_size)(conv4)
    model = tf.keras.models.Model(inputs, outputs)
    return model

In [11]:
input_shape = (patch_size, patch_size, 3)
espcn_model_with_skip_and_attention = build_espcn_model_with_skip_connections(input_shape, scale_size)
espcn_model_with_skip_and_attention.compile(optimizer='adam', loss='mse', metrics=[psnr])

c:\Users\91995\anaconda3\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [12]:
espcn_model_with_skip_and_attention.fit(lr_patches, hr_patches, epochs=20)

Epoch 1/20
4754/4754 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - loss: 0.0088 - psnr: 26.5483
Epoch 2/20
4754/4754 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - loss: 8.0092e-04 - psnr: 34.3653
Epoch 3/20
4754/4754 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - loss: 7.4346e-04 - psnr: 35.1971
Epoch 4/20
4754/4754 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - loss: 7.2902e-04 - psnr: 35.4618
Epoch 5/20
4754/4754 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - loss: 7.1970e-04 - psnr: 35.6877
Epoch 6/20
4754/4754 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - loss: 7.1531e-04 - psnr: 35.7518
Epoch 7/20
4754/4754 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - loss: 7.1270e-04 - psnr: 35.8301
Epoch 8/20
4754/4754 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - loss: 7.1267e-04 - psnr: 35.8017
Epoch 9/20
4754/4754 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - loss: 7.0806e-04 - psnr: 35.8799
Epoch 10/20
4754/4754 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - loss: 7.0695e-04 - psnr: 35.8601
Epoch 11/20
4754/4754 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - loss: 7.0687e-04 - psnr: 35.8560
Epoch 12/20
